In [ ]:
!pip install pandas
!pip install scikit-learn
!pip install pyarrow

In [ ]:
!pip freeze | grep scikit-learn

In [ ]:
!python -V

In [ ]:
import pickle
import pandas as pd
import numpy as np

In [ ]:

with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [ ]:
def get_data(year, month):
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet'
    df = pd.read_parquet(url)
    return df

In [ ]:
categorical = ['PULocationID', 'DOLocationID']

def engineer_data(df, year, month):
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
    
    return df


In [ ]:
df = get_data(2023, 3)
df = engineer_data(df, 2023, 3)

In [ ]:

dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [ ]:
print(np.std(y_pred))

In [ ]:
df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'predicted_duration': y_pred,
})

df_result.to_parquet(
    "results.parquet",
    engine='pyarrow',
    compression=None,
    index=False
)